In [1]:
import pandas as pd
import numpy as np

from sklearn.ensemble import GradientBoostingClassifier
from sklearn.model_selection import RandomizedSearchCV
from sklearn.model_selection import StratifiedKFold

from skopt import BayesSearchCV
from skopt.space import Real, Integer

from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import matplotlib.pyplot as plt
from sklearn.metrics import ConfusionMatrixDisplay
import warnings, os, joblib
from sklearn.utils.class_weight import compute_sample_weight

####  Load Training and Testing Dataset

In [2]:
train_df = pd.read_csv("./dataset/Clean_CICDDOS_Train.csv")
test_df = pd.read_csv("./dataset/Clean_CICDDOS_Test.csv")

print("Train Shape:", train_df.shape)
print("Test Shape:", test_df.shape)

Train Shape: (659455, 67)
Test Shape: (120199, 67)


In [3]:
train_df.columns

Index(['Protocol', 'Flow_Duration', 'Total_Fwd_Packets',
       'Total_Backward_Packets', 'Total_Length_of_Fwd_Packets',
       'Total_Length_of_Bwd_Packets', 'Fwd_Packet_Length_Max',
       'Fwd_Packet_Length_Min', 'Fwd_Packet_Length_Mean',
       'Fwd_Packet_Length_Std', 'Bwd_Packet_Length_Max',
       'Bwd_Packet_Length_Min', 'Bwd_Packet_Length_Mean',
       'Bwd_Packet_Length_Std', 'Flow_Bytes/s', 'Flow_Packets/s',
       'Flow_IAT_Mean', 'Flow_IAT_Std', 'Flow_IAT_Max', 'Flow_IAT_Min',
       'Fwd_IAT_Total', 'Fwd_IAT_Mean', 'Fwd_IAT_Std', 'Fwd_IAT_Max',
       'Fwd_IAT_Min', 'Bwd_IAT_Total', 'Bwd_IAT_Mean', 'Bwd_IAT_Std',
       'Bwd_IAT_Max', 'Bwd_IAT_Min', 'Fwd_PSH_Flags', 'Fwd_Header_Length',
       'Bwd_Header_Length', 'Fwd_Packets/s', 'Bwd_Packets/s',
       'Min_Packet_Length', 'Max_Packet_Length', 'Packet_Length_Mean',
       'Packet_Length_Std', 'Packet_Length_Variance', 'SYN_Flag_Count',
       'RST_Flag_Count', 'ACK_Flag_Count', 'URG_Flag_Count', 'CWE_Flag_Count',
      

In [5]:
train_df.describe().T

,count,mean,std,min,25%,50%,75%,max
Protocol,659455.0,1.511424e+01,4.146729e+00,0.0,17.0,17.0,17.0,1.700000e+01
Flow_Duration,659455.0,1.438022e+06,9.494608e+06,0.0,1.0,1.0,151.0,1.200000e+08
Total_Fwd_Packets,659455.0,8.917503e+00,2.515824e+02,1.0,2.0,2.0,4.0,9.939500e+04
Total_Backward_Packets,659455.0,6.714484e-02,9.855134e-01,0.0,0.0,0.0,0.0,2.870000e+02
Total_Length_of_Fwd_Packets,659455.0,3.873926e+03,1.144862e+04,0.0,458.0,1398.0,2944.0,1.760000e+05
...,...,...,...,...,...,...,...,...
Idle_Std,659455.0,9.208356e+04,8.586330e+05,0.0,0.0,0.0,0.0,4.533786e+07
Idle_Max,659455.0,5.227058e+05,3.626142e+06,0.0,0.0,0.0,0.0,9.409758e+07
Idle_Min,659455.0,3.361862e+05,2.431835e+06,0.0,0.0,0.0,0.0,9.409758e+07
Inbound,659455.0,9.973797e-01,5.112224e-02,0.0,1.0,1.0,1.0,1.000000e+00


In [ ]:
train_df['Label'].unique()

array([ 1,  0,  2,  3,  5,  4,  6,  7, 10,  8,  9, 11, 12])

In [12]:
train_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 659455 entries, 0 to 659454
Data columns (total 67 columns):
 #   Column                       Non-Null Count   Dtype  
---  ------                       --------------   -----  
 0   Protocol                     659455 non-null  int64  
 1   Flow_Duration                659455 non-null  int64  
 2   Total_Fwd_Packets            659455 non-null  int64  
 3   Total_Backward_Packets       659455 non-null  int64  
 4   Total_Length_of_Fwd_Packets  659455 non-null  float64
 5   Total_Length_of_Bwd_Packets  659455 non-null  float64
 6   Fwd_Packet_Length_Max        659455 non-null  float64
 7   Fwd_Packet_Length_Min        659455 non-null  float64
 8   Fwd_Packet_Length_Mean       659455 non-null  float64
 9   Fwd_Packet_Length_Std        659455 non-null  float64
 10  Bwd_Packet_Length_Max        659455 non-null  float64
 11  Bwd_Packet_Length_Min        659455 non-null  float64
 12  Bwd_Packet_Length_Mean       659455 non-null  float64
 13 

In [4]:
train_cols = set(train_df.columns)
test_cols = set(test_df.columns)

print(f"In Train but not Test: {train_cols - test_cols}")
print(f"In Test but not Train: {test_cols - train_cols}")

In Train but not Test: set()
In Test but not Train: set()


#### separate features and labels

In [6]:
X_train = train_df.drop("Label", axis=1)
y_train = train_df["Label"]

X_test = test_df.drop("Label", axis=1)
y_test = test_df["Label"]

X_train.isnull().sum().sum(), X_test.isnull().sum().sum()

(np.int64(0), np.int64(0))

#### Strong feature stabilization

In [7]:
# replace inf
X_train = X_train.replace([np.inf, -np.inf], np.nan)
X_test = X_test.replace([np.inf, -np.inf], np.nan)

X_train.isnull().sum().sum(), X_test.isnull().sum().sum()

(np.int64(0), np.int64(0))

In [8]:
X_train = X_train.clip(0, 1e5)
X_test = X_test.clip(0, 1e5)

X_train.isnull().sum().sum(), X_test.isnull().sum().sum()

(np.int64(0), np.int64(0))

#### Handle the class imbalance in the Model

In [9]:
# calculate wights to balance the 13 classes
sample_weights = compute_sample_weight(class_weight='balanced', y=y_train)


#### Gradient Boosting Model

In [14]:
gb_model = GradientBoostingClassifier(
    n_estimators=150, # number of trees (More tree better model but slow)
    learning_rate=0.05, # shrinkage factor per tree
    max_depth=4,        # max depth of each individual tree
    subsample=0.8,     # use 80% data per tree (adds variance diversity)
    min_samples_leaf=20,  # min samples required in a leaf
    max_features='sqrt',      # consider sqrt(n_features) at each split → faster
    validation_fraction=0.1,
    n_iter_no_change=10, # early stopping
    # tol=0.01,  # stop if loss doesn't imporve by this much
    random_state=42,
    verbose=1          # print progress
)

In [15]:
print("Training Model ...")
gb_model.fit(X_train, y_train, sample_weight=sample_weights)
print("Model Training Completed")

Training Model ...
      Iter       Train Loss      OOB Improve   Remaining Time 
         1           2.2036           0.3558           32.17m
         2           1.9688           0.2436           29.94m
         3           1.7954           0.1577           28.30m
         4           1.6621           0.1401           27.20m
         5           1.5399           0.1210           28.01m
         6           1.4468           0.1051           27.87m
         7           1.3592           0.0922           26.75m
         8           1.2811           0.0587           25.94m
         9           1.2143           0.0712           25.23m
        10           1.1532           0.0491           24.60m
        20           0.7975          -0.0191           23.13m
        30           0.6498          -0.0085           20.70m
        40           0.5820           0.0172           18.67m
        50           0.5358          -0.0235           17.72m
        60           0.5155          -0.0043      

#### Model Evaluation

In [16]:
y_pred = gb_model.predict(X_test)

In [17]:
print("Accuracy:", accuracy_score(y_test, y_pred))

Accuracy: 0.5843975407449313


In [18]:
label_encoder = joblib.load('./models/LabelEncoder.pkl')
class_names = list(label_encoder.classes_)
print("\nClassification Report")
print(classification_report(y_test, y_pred, target_names=class_names))


Classification Report
              precision    recall  f1-score   support

      BENIGN       0.05      0.98      0.10       835
         DNS       0.00      0.00      0.00         0
        LDAP       0.99      0.46      0.63     18133
       MSSQL       0.97      0.95      0.96     20032
         NTP       0.00      0.00      0.00         0
     NetBIOS       1.00      1.00      1.00     21804
        SNMP       0.00      0.00      0.00         0
        SSDP       0.00      0.00      0.00         0
         Syn       1.00      0.09      0.16     36458
        TFTP       0.00      0.00      0.00         0
         UDP       1.00      0.74      0.85     22885
      UDPLag       0.00      0.02      0.00        52
     WebDDoS       0.00      0.00      0.00         0

    accuracy                           0.58    120199
   macro avg       0.38      0.33      0.28    120199
weighted avg       0.99      0.58      0.65    120199



d:\NewUser\Programs\miniconda3\Lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
d:\NewUser\Programs\miniconda3\Lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
d:\NewUser\Programs\miniconda3\Lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


In [70]:

print("\nConfusion Matrix")
confusion_matrix(y_test, y_pred)


Confusion Matrix


array([[  684,     0,     0,     3,     4,    11,     0,     0,     7,
            0,     6,     8,   112],
       [    0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0],
       [    0,  2228, 12783,   128,     3,     2,  2987,     0,     0,
            1,     0,     0,     1],
       [    1,    53,   183, 18897,    10,     1,   663,     0,     0,
           48,   174,     1,     1],
       [    0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0],
       [    1,     0,     0,     4,     2, 21761,     0,    16,     0,
            0,    20,     0,     0],
       [    0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0],
       [    0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0],
       [15016,     0,     0,    25,     0,     0,     0,     0, 20880,
            0,     3,   526,     8],
       [    0,     0,     0,

In [19]:
fig, ax = plt.subplots(figsize=(12, 10))
cm = confusion_matrix(y_test, y_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=class_names)
disp.plot(ax=ax, colorbar=True, xticks_rotation=45)
ax.set_title("Gradient Boost — Confusion Matrix (CIC-DDoS 2019)", fontsize=13)
plt.tight_layout()
cm_path = os.path.join('./results/confusion_matrix2.png')
plt.savefig(cm_path, dpi=150, bbox_inches='tight')
plt.close()
print(f"\n  ✓ Confusion matrix saved → {cm_path}")


  ✓ Confusion matrix saved → ./results/confusion_matrix2.png


#### RandomizedSearchCV

In [20]:
param_dist = {
    
    "n_estimators": [100, 200, 300, 400, 500],
    
    "learning_rate": [0.01, 0.05, 0.1, 0.2],
    
    "max_depth": [3, 4, 5, 6, 7],
    
    "min_samples_split": [2, 5, 10],
    
    "min_samples_leaf": [1, 2, 4],
    
    "subsample": [0.6, 0.8, 1.0]
}

random_search = RandomizedSearchCV(

    estimator=GradientBoostingClassifier(random_state=42),

    param_distributions=param_dist,

    n_iter=30,

    scoring="accuracy",

    cv=StratifiedKFold(n_splits=5),

    verbose=2,

    random_state=42,

    n_jobs=-1
)

In [21]:
random_search.fit(X_train, y_train)

Fitting 5 folds for each of 30 candidates, totalling 150 fits


MemoryError: Unable to allocate 181. MiB for an array with shape (45, 527564) and data type float64

In [ ]:
print("Best Parameters:", random_search.best_params_)

- best model prediction

In [ ]:
best_random_model = random_search.best_estimator_

y_pred_random = best_random_model.predict(X_test)

In [ ]:
print("Random Search Accuracy:", accuracy_score(y_test, y_pred_random))

print(classification_report(y_test, y_pred_random))

#### BayesSearchCV

In [ ]:
search_space = {

    "n_estimators": Integer(100, 500),

    "learning_rate": Real(0.01, 0.2),

    "max_depth": Integer(3, 7),

    "min_samples_split": Integer(2, 10),

    "min_samples_leaf": Integer(1, 4),

    "subsample": Real(0.6, 1.0)
}

bayes_search = BayesSearchCV(

    estimator=GradientBoostingClassifier(random_state=42),

    search_spaces=search_space,

    n_iter=30,

    scoring="accuracy",

    cv=StratifiedKFold(n_splits=5),

    verbose=2,

    random_state=42,

    n_jobs=-1
)

In [ ]:
bayes_search.fit(X_train, y_train)

In [ ]:
print("Best Parameters:", bayes_search.best_params_)

In [ ]:
best_bayes_model = bayes_search.best_estimator_

y_pred_bayes = best_bayes_model.predict(X_test)

In [ ]:
print("Bayesian Search Accuracy:", accuracy_score(y_test, y_pred_bayes))

print(classification_report(y_test, y_pred_bayes))

#### Compare Bothe Models

In [ ]:
random_acc = accuracy_score(y_test, y_pred_random)

bayes_acc = accuracy_score(y_test, y_pred_bayes)

print("Random Search Accuracy:", random_acc)
print("Bayesian Search Accuracy:", bayes_acc)

In [ ]:
if bayes_acc > random_acc:
    best_model = best_bayes_model
    print("Bayesian model selected")
else:
    best_model = best_random_model
    print("Random Search model selected")

#### Save model

In [ ]:
import joblib

joblib.dump(best_model, "ddos_gradient_boost_model.pkl")

In [ ]:
# model = joblib.load("ddos_gradient_boost_model.pkl")